# Comparing photometry
This notebook is intended to validate different photometric measurements by comparing their completeness, magnitudes, and residuals against each other.
We compare against both different methods (psf, SE, photutils, SEP) and data sources (GTC, PanStarrs, and DESI LSS). 

In general, the residuals and results appear to be well-behaved. However, uncertainties appear to typically be underestimated by a factor of ~2. The methods used to subtract the background additionally bias the magnitudes of faint sources, adding additional uncertainty to these comparisons. 

However, we conclude that the photometry used here is adequate at minimum for detection of objects.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya


In [ ]:
from copy import copy

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

In [ ]:
from yasone import photometry as phot_utils
from yasone import analysis as ana_utils
from yasone import selection as filter_utils

from yasone.plotting import plot_mag_comparison, plot_mag_residual_hist, plot_mag_2d_residual
from yasone.photometry import outer_join_xmatch

filter_utils.load_isochrones()

In [ ]:
import sys
sys.path.append("../photometry/")
import calc_panstarrs_offset

# Utilities

In [ ]:
def get_panstarrs(objname):
    panstarrs = calc_panstarrs_offset.get_panstarrs(objname)
    panstarrs.rename_columns(
        ["G_MAG", "R_MAG", "I_MAG", "G_MAG_ERR", "R_MAG_ERR", "I_MAG_ERR"],
        ["G_MAG_PS", "R_MAG_PS", "I_MAG_PS", "G_MAG_PS_ERR", "R_MAG_PS_ERR", "I_MAG_PS_ERR"],
    )
    return panstarrs

In [ ]:
def filter_edges(cat, obj, xi_eta_max=3):
    """
    Filters stars in the catalogue to be within `xi_eta_max` of the centre of the provided `obj`. 
    Useful for comparing completeness on the same footprint
    """
    xi, eta = ana_utils.to_tangent(ana_utils.to_coords(cat), ana_utils.get_coord0(obj))
    return cat[(np.abs(60*xi) < xi_eta_max)
        & (np.abs(eta*60) < xi_eta_max) ]

In [ ]:
def plot_completeness_vs_mag(
    table,
    mag_ref_col,
    mag_det_col,
    bins=20,
    mag_range=None,
    **kwargs
):
    """
    Plot completeness as a function of reference magnitude.

    Parameters
    ----------
    table : astropy.table.Table
        Crossmatched table with masked magnitude columns.
    mag_ref_col : str
        Column name of reference magnitude (defines denominator).
    mag_det_col : str
        Column name of detection magnitude (defines numerator).
    bins : int or array
        Number of bins or bin edges.
    mag_range : tuple or None
        (min, max) magnitude range.
    """

    mag_ref = table[mag_ref_col]
    mag_det = table[mag_det_col]

    # Valid reference sources (denominator)
    ref_valid = ~mag_ref.mask

    # Detected sources (numerator)
    detected = (~mag_ref.mask) & (~mag_det.mask)

    # Extract reference magnitudes for binning
    mag_values = mag_ref[ref_valid].data

    if mag_range is None:
        mag_min, mag_max = np.nanmin(mag_values), np.nanmax(mag_values)
    else:
        mag_min, mag_max = mag_range

    # Define bins
    bin_edges = np.linspace(mag_min, mag_max, bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    # Count per bin
    N_ref, _ = np.histogram(mag_values, bins=bin_edges)

    # Count detected per bin
    detected_mags = mag_ref[detected].data
    N_det, _ = np.histogram(detected_mags, bins=bin_edges)

    # Compute completeness
    with np.errstate(divide="ignore", invalid="ignore"):
        completeness = N_det / N_ref
        completeness[N_ref == 0] = np.nan

    # Plot
    plt.plot(bin_centers, completeness, **kwargs,)
    plt.xlabel(f"{mag_ref_col}")
    plt.ylabel(f"Completeness in {mag_det_col}")
    plt.ylim(0, 1.05)

    return bin_centers, completeness

In [ ]:
def get_lss_cat(objname):
    cat_lss = Table.read(f"../survey_data/{objname}_lss10.csv")
    cat_lss.rename_columns(["mag_g", "mag_r", "mag_i"], ["G_MAG_LSS", "R_MAG_LSS", "I_MAG_LSS"])


    for filt in ["G", "R", "I"]:
        cat_lss[f"{filt}_MAG_LSS"] = np.ma.masked_array(cat_lss[f"{filt}_MAG_LSS"], ~np.isfinite(cat_lss[f"{filt}_MAG_LSS"]))

    xi, eta = ana_utils.to_tangent(ana_utils.to_coords(cat_lss), ana_utils.get_coord0(objname))
    cat_lss["xi"] = xi * 60 * u.arcmin
    cat_lss["eta"] = eta * 60 * u.arcmin
    
    cat_lss["G_MAG_LSS_ERR"] = 2.5 / np.log(10) / cat_lss["snr_i"]
    cat_lss["R_MAG_LSS_ERR"] = 2.5 / np.log(10) / cat_lss["snr_r"]
    cat_lss["I_MAG_LSS_ERR"] = 2.5 / np.log(10) / cat_lss["snr_g"]

    return cat_lss


In [ ]:
def detection_plot_gr(cat, params, xi=1.5/60, eta=-1.5/60, depth_sigma=5):
    subsets = filter_utils.select_subsets(cat, params, xi=xi, eta=eta)
    results = filter_utils.catalog_stats(cat, params).iloc[0, :]

    fig, axs = plt.subplots(2, 3, figsize=(6, 4))

    plt.sca(axs[0][0])
    filter_utils.plot_tangent(cat, s=0.3, lw=0, color="k")
    plt.title("all")

    plt.sca(axs[1][0])
    filter_utils.plot_selected_points(subsets, params, xi=xi, eta=eta)
    plt.gca().invert_xaxis()
    
    plt.sca(axs[1][1])
    filter_utils.plot_gr_background(subsets, params)


    plt.sca(axs[0][1])
    filter_utils.plot_gr_centre(subsets, params)


    plt.tight_layout()


In [ ]:
def plot_mag_shifts(cat, rcol_1, icol_1, rcol_2, icol_2):
    x1 = cat[rcol_1] - cat[icol_1]
    y1 = cat[rcol_1]
    x2 = cat[rcol_2] - cat[icol_2]
    y2 = cat[rcol_2]

    filt = filter_utils.filt_finite(x2 - x1) & filter_utils.filt_finite(y2 - y1) 
    print(np.sum(filt), "matched points, out of", len(x1))
    
    plt.scatter(x1, y1, s=1 + ~filt, lw=0)
    plt.quiver(x1[filt].data, y1[filt].data, (x2-x1)[filt].data, (y2-y1)[filt].data, lw=0.3, alpha=0.5, scale_units='xy', angles='xy', scale=1)
    plt.scatter(x2, y2, s=0.5 + ~filt, lw=0)


In [ ]:
def detection_shifts(cat, params, xi=1.5/60, eta=-1.5/60, depth_sigma=5,
                       gcol_1="G_MAG", rcol_1="R_MAG", icol_1="I_MAG", 
                     gcol_2="G_MAG_2", rcol_2="R_MAG_2", icol_2="I_MAG_2",
                      ):
    subsets = filter_utils.select_subsets(cat, params, xi=xi, eta=eta)

    gr_err =  filter_utils.get_gr_err(cat, params)
    ri_err =  filter_utils.get_ri_err(cat, params)

    fig, axs = plt.subplots(2, 3, figsize=(6, 4))

    # gr
    x0 = cat[gcol_1] - cat[rcol_1]
    x1 = cat[gcol_2] - cat[rcol_2]

    filt1 = filter_utils.filt_finite(x0)
    filt2 = filter_utils.filt_finite(x1) 

    plt.sca(axs[0][0])
    filter_utils.plot_tangent(cat[filt1], s=1, lw=0, color="C0")
    filter_utils.plot_tangent(cat[filt2], s=0.5, lw=0, color="C1")
    plt.title("all")

    plt.sca(axs[0][1])
    filter_utils.gr_axis()
    plot_mag_shifts(subsets["centre_all"], gcol_1, rcol_1, gcol_2, rcol_2)
    filter_utils.plot_iso_gr(params, gr_err)

    plt.sca(axs[0][2])
    filter_utils.gr_axis()

    plot_mag_shifts(subsets["background_all"], gcol_1, rcol_1, gcol_2, rcol_2)
    filter_utils.plot_iso_gr(params, gr_err)

    
    # ri
    x0 = cat[rcol_1] - cat[icol_1]
    x1 = cat[rcol_2] - cat[icol_2]

    filt1 = filter_utils.filt_finite(x0)
    filt2 = filter_utils.filt_finite(x1) 

    plt.sca(axs[1][0])
    filter_utils.plot_tangent(cat[filt1], s=1, lw=0, color="C0")
    filter_utils.plot_tangent(cat[filt2], s=0.5, lw=0, color="C1")
    plt.title("all")

    plt.sca(axs[1][1])
    filter_utils.ri_axis()
    plot_mag_shifts(subsets["centre_all"], rcol_1, icol_1, rcol_2, icol_2)
    filter_utils.plot_iso_ri(params, ri_err)


    plt.sca(axs[1][2])
    filter_utils.ri_axis()

    plot_mag_shifts(subsets["background_all"], rcol_1, icol_1, rcol_2, icol_2)
    filter_utils.plot_iso_ri(params, ri_err)

    

    plt.tight_layout()


In [ ]:
def detection_shift_gr(cat, params, xi=1.5/60, eta=-1.5/60, depth_sigma=5,
                       rcol_1="G_MAG", icol_1="R_MAG", rcol_2="G_MAG_2", icol_2="R_MAG_2",
                      ):
    subsets = filter_utils.select_subsets(cat, params, xi=xi, eta=eta)

    coord0 = ana_utils.get_coord0(params.objname)
    coord1 = SkyCoord(ra=coord0.ra.value + xi/np.cos(np.deg2rad(coord0.dec.value)), dec=coord0.dec.value + eta, unit=u.deg)
    cat_cen = ana_utils.select_centre(cat, coord0, params.r_cen * u.arcmin)
    cat_bkg = ana_utils.select_centre(cat, coord1, params.r_cen * u.arcmin)

    x0 = cat[rcol_1] - cat[icol_1]
    x1 = cat[rcol_2] - cat[icol_2]

    filt1 = filter_utils.filt_finite(x0)
    filt2 = filter_utils.filt_finite(x1) 

    fig, axs = plt.subplots(1, 3, figsize=(6, 4))

    plt.sca(axs[0])
    filter_utils.plot_tangent(cat[filt1], s=1, lw=0, color="C0")
    filter_utils.plot_tangent(cat[filt2], s=0.5, lw=0, color="C1")
    plt.title("all")

    plt.sca(axs[1])
    filter_utils.gr_axis()
    plot_mag_shifts(cat_cen, rcol_1, icol_1, rcol_2, icol_2)
    filter_utils.plot_iso_gr(params, filter_utils.get_gr_err(cat, params))
    
    plt.sca(axs[2])
    filter_utils.gr_axis()
    filter_utils.plot_iso_gr(params, filter_utils.get_gr_err(cat, params))

    plot_mag_shifts(cat_bkg, rcol_1, icol_1, rcol_2, icol_2)


    plt.tight_layout()


In [ ]:
def plot_xmatch_residuals(cat_xmatch):
    plt.hist((cat_xmatch["ra_1"] - cat_xmatch["ra_2"])*3600, histtype="step", label="ra")
    plt.hist((cat_xmatch["dec_1"] - cat_xmatch["dec_2"])*3600, histtype="step", label="dec")

    plt.yscale("log")
    plt.xlabel("xmatch residuals")
    plt.legend()

In [ ]:
def get_default_params(objname):
    obs_props = ana_utils.get_obs_props(objname)
    return filter_utils.filter_params(
        dm = obs_props["dm"], 
        iso_fe_h = obs_props["iso_fe_h"],
        iso_log_age = np.log10(obs_props["iso_age"]),
        objname = objname, 
        mode="gri", 
        r23_max_sigma=None,
        r_cen = 40/60,
        iso_width=0.15
    )
    

In [ ]:
def fix_coordinates(cat_matched, objname):

    cat_matched["ra"] = np.ma.mean([cat_matched["ra_1"], cat_matched["ra_2"]], axis=0)
    cat_matched["dec"] = np.ma.mean([cat_matched["dec_1"], cat_matched["dec_2"]], axis=0)
    
    xi, eta = ana_utils.to_tangent(ana_utils.to_coords(cat_matched), ana_utils.get_coord0(objname))
    cat_matched["xi"] = xi * 60 * u.arcmin
    cat_matched["eta"] = eta * 60 * u.arcmin

    for filt in ["G", "R", "I"]:
        if filt + "_MAG_1" in cat_matched.colnames:
            cat_matched.rename_column(filt + "_MAG_1", filt + "_MAG")
            cat_matched.rename_column(filt + "_MAG_ERR_1", filt + "_MAG_ERR")

    cat_matched["INDEX"] = np.arange(len(cat_matched))
    return cat_matched

# Yasone 1

In [ ]:
cat = ana_utils.read_catalogue("yasone1", catname="allcolours", shiftname="allcolours_panstarrs_shift")

cat_ps = get_panstarrs("yasone1")
cat_lss = get_lss_cat("yasone1")
cat = filter_edges(cat, "yasone1")
cat_ps = filter_edges(cat_ps, "yasone1")
cat_lss = filter_edges(cat_lss, "yasone1")

In [ ]:
cat_x_lss = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
cat_x_ps = phot_utils.outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_x_lss, "yasone1")
fix_coordinates(cat_x_ps, "yasone1")

## Comparing the completeness of each catalogue

In [ ]:
plot_completeness_vs_mag(cat_x_ps, "G_MAG_PS", "G_MAG", bins=10, label="G")
plot_completeness_vs_mag(cat_x_ps, "R_MAG_PS", "R_MAG", bins=10, label="R")
plot_completeness_vs_mag(cat_x_ps, "I_MAG_PS", "I_MAG", bins=10, label="I")
plt.legend()
plt.xlabel("PS magnitude")
plt.ylabel("completeness in GTC")

In [ ]:
plot_completeness_vs_mag(cat_x_lss, "G_MAG_LSS", "G_MAG", bins=10, label="G")
plot_completeness_vs_mag(cat_x_lss, "R_MAG_LSS", "R_MAG", bins=10, label="R")
plt.legend()
plt.xlabel("LSS magnitude")
plt.ylabel("completeness in GTC")

In [ ]:
plot_completeness_vs_mag(cat_x_ps, "G_MAG", "G_MAG_PS", bins=10, label="G (PS)")
plot_completeness_vs_mag(cat_x_ps, "R_MAG", "R_MAG_PS", bins=10, label="R (PS)")
plot_completeness_vs_mag(cat_x_ps, "I_MAG", "I_MAG_PS", bins=10, label="I (PS)")

plot_completeness_vs_mag(cat_x_lss, "G_MAG", "G_MAG_LSS", bins=10, label="G (LSS)")
plot_completeness_vs_mag(cat_x_lss, "R_MAG", "R_MAG_LSS", bins=10, label="R (LSS)")
plt.legend()
plt.xlabel("GTC magnitude")


In [ ]:
plot_xmatch_residuals(cat_x_lss)

In [ ]:
plot_xmatch_residuals(cat_x_ps)

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat["R_MAG"], bins, histtype="step", label="osiris")

plt.hist(cat_lss["R_MAG_LSS"], bins, histtype="step", label="LSS")
plt.hist(cat_ps["R_MAG_PS"], bins, histtype="step", label="PS")
plt.xlabel("r mag")
plt.ylabel("count")
arya.Legend(-1)

## Residuals between catalogues

In [ ]:
plot_mag_comparison(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_LSS_ERR",
                    filters=["g", "r"],
                    ref_label="osiris",
                    cmp_label="LSS"
                    
                   )

In [ ]:
plot_mag_2d_residual(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    filters=["g", "r"])


In [ ]:
plot_mag_comparison(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_2d_residual(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                   )

In [ ]:
plot_mag_residual_hist(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], 
                       xlim=(-10, 10),
                       sigma_scale=2
                   )

In [ ]:
plot_mag_residual_hist(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_LSS_ERR", 
                    filters=["g", "r"], 
                       xlim=(-10, 10),
                       sigma_scale=0
                   )

In [ ]:
params = get_default_params("yasone1")


In [ ]:
detection_shifts(cat_x_ps, params, gcol_2="G_MAG_PS", rcol_2="R_MAG_PS", icol_2="I_MAG_PS")

In [ ]:
cat_ps_x_lss = outer_join_xmatch(cat_ps, cat_lss)
fix_coordinates(cat_ps_x_lss, "yasone1")


In [ ]:
params_new = copy(params)
params_new.flags_iso_max = None
params_new.flags_max = None
params_new.flags_weight_max = None
params_new.detection_sigma = None

cat_ps_x_lss["I_MAG"] = cat_ps_x_lss["I_MAG_LSS"]
cat_ps_x_lss["R_MAG"] = cat_ps_x_lss["R_MAG_LSS"]
cat_ps_x_lss["G_MAG"] = cat_ps_x_lss["G_MAG_PS"]
cat_ps_x_lss["I_MAG_ERR"] = cat_ps_x_lss["I_MAG_LSS_ERR"]
cat_ps_x_lss["R_MAG_ERR"] = cat_ps_x_lss["R_MAG_LSS_ERR"]
cat_ps_x_lss["G_MAG_ERR"] = cat_ps_x_lss["G_MAG_PS_ERR"]
cat_ps_x_lss["INDEX"] = np.arange(len(cat_ps_x_lss))


In [ ]:
detection_shifts(cat_x_ps, params, gcol_2="G_MAG_PS", rcol_2="R_MAG_PS", icol_2="I_MAG_PS")

## Comparing photometry methods

### Forced Photometry

In [ ]:
# cat_sep = ana_utils.read_catalogue("yasone1", catname="julen")
# cat_julen = ana_utils.read_catalogue("yasone1", catname="allcolours_julen_stack")

cat_forced = ana_utils.read_catalogue("yasone1", catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat_psf = ana_utils.read_catalogue("yasone1", catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_forced = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")

cat_small_bkg = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_small_bkg", shiftname="allcolours_panstarrs_shift")
cat_large_bkg = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_large_bkg", shiftname="allcolours_panstarrs_shift")
cat_small_bkg_filt = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_small_bkg_filt", shiftname="allcolours_panstarrs_shift")
cat_large_bkg_filt = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_large_bkg_filt", shiftname="allcolours_panstarrs_shift")


In [ ]:
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_3")
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_LB_3")

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF_ANA")

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=1
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                   )

In [ ]:
detection_shifts(cat_forced, params_new,  gcol_2="G_MED_MAG_APER_3", rcol_2="R_MED_MAG_APER_3", icol_2="I_MED_MAG_APER_3")


### Forced photometry (local background)

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="local bkg"
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                   )

In [ ]:
detection_shifts(cat_forced, params_new, gcol_1="G_MED_MAG_APER_3", rcol_1="R_MED_MAG_APER_3", icol_1="I_MED_MAG_APER_3", 
                 gcol_2="G_MED_MAG_APER_LB_3", rcol_2="R_MED_MAG_APER_LB_3", icol_2="I_MED_MAG_APER_LB_3")


In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"],
                   )

### PSF

In [ ]:
plot_mag_comparison(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"], ref_label="aperture stack", cmp_label="forced PSF"
                   )

In [ ]:
plot_mag_residual_hist(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=1
                   )

In [ ]:
detection_shifts(cat_psf, params_new, gcol_2="G_MED_MAG_PSF", rcol_2="R_MED_MAG_PSF", icol_2="I_MED_MAG_PSF")


### SEP

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_sep, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_joined, "yasone1")

In [ ]:
cat_joined["G_MAG_2"] += 0.1
cat_joined["R_MAG_2"] += 0.2
cat_joined["I_MAG_2"] += 0.1

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
params_new = copy(params)
params_new.detection_sigma = None

In [ ]:
detection_shifts(cat_joined, params_new, )

### Julen's data / process

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_julen, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_joined, "yasone1");

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
params_new = copy(params)
params_new.flags_max = None
params_new.flags_weight_max = None
params_new.flags_iso_max = None
params_new.detection_sigma = None

In [ ]:
detection_shifts(cat_joined, params_new,)

### Photometry ssystems

In [ ]:
for tab in [cat_forced, cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    filter_utils.calibrate_mag_col(tab, "MED_MAG_APER_3")

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    fig, axs = plt.subplots(1, 3, figsize=(6, 2), sharex=True, sharey=True)

    cmax = 1
    plt.sca(axs[0])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat_forced["G_MED_MAG_APER_3"] - cat2["G_MED_MAG_APER_3"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=0.3)

    plt.sca(axs[1])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat_forced["R_MED_MAG_APER_3"] - cat2["R_MED_MAG_APER_3"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=0.3)

    plt.sca(axs[2])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat_forced["I_MED_MAG_APER_3"] - cat2["I_MED_MAG_APER_3"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=0.3)
    plt.colorbar(label="flux difference")

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    fig, axs = plt.subplots(1, 3, figsize=(6, 2), sharex=True, sharey=True)

    cmax = 10_000
    plt.sca(axs[0])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat_forced["G_aperture_sum_3_median"] - cat2["G_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)

    plt.sca(axs[1])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat_forced["R_aperture_sum_3_median"] - cat2["R_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)

    plt.sca(axs[2])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat_forced["I_aperture_sum_3_median"] - cat2["I_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)
    plt.colorbar(label="flux difference")

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    cat_joined = fix_coordinates(outer_join_xmatch(cat_forced, cat2), "yasone1")
    detection_shifts(cat_joined, params_new, gcol_1="G_MED_MAG_APER_3_1", rcol_1="R_MED_MAG_APER_3_1",  icol_1="I_MED_MAG_APER_3_1",
                 gcol_2="G_MED_MAG_APER_3_2", rcol_2="R_MED_MAG_APER_3_2", icol_2="I_MED_MAG_APER_3_2")


In [ ]:
import yasone

In [ ]:
# this plot validates how we plot the uncertainties...
cat_joined = phot_utils.outer_join_xmatch(cat_forced, cat_large_bkg, ra1="ra", dec1="dec", ra2="ra", dec2="dec")

magerr = np.sqrt(cat_joined["G_MED_MAG_APER_3_ERR_2"]**2 + cat_joined["G_MAG_ERR_1"]**2)
plt.scatter(cat_joined["G_MAG_1"], magerr, s=1, lw=0, alpha=0.1)
yasone.plotting.plot_sliding_window_mag_error(plt.gca(), cat_joined["G_MAG_1"], magerr)

plt.ylim(0, 1)

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    cat_joined = phot_utils.outer_join_xmatch(cat_forced, cat2, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
    
    plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MED_MAG_APER_3_1", mag_cmp_fmt="{FILT}_MED_MAG_APER_3_2", 
                        mag_ref_err_fmt="{FILT}_MED_MAG_APER_3_ERR_1", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR_2", 
    
                        filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18))
    plt.show()

# Yasone 2

In [ ]:
cat = ana_utils.read_catalogue("yasone2", catname="allcolours", shiftname="allcolours_panstarrs_shift")

cat_ps = get_panstarrs("yasone2")
cat_lss = get_lss_cat("yasone2")
cat = filter_edges(cat, "yasone2")
cat_ps = filter_edges(cat_ps, "yasone2")
cat_lss = filter_edges(cat_lss, "yasone2")

In [ ]:
cat_x_lss = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
cat_x_ps = phot_utils.outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_x_lss, "yasone2")
fix_coordinates(cat_x_ps, "yasone2")

## Comparing the completeness of each catalogue

In [ ]:
plot_completeness_vs_mag(cat_x_ps, "G_MAG_PS", "G_MAG", bins=20, label="G")
plot_completeness_vs_mag(cat_x_ps, "R_MAG_PS", "R_MAG", bins=20, label="R")
plot_completeness_vs_mag(cat_x_ps, "I_MAG_PS", "I_MAG", bins=20, label="I")
plt.legend()
plt.xlabel("PS magnitude")
plt.ylabel("completeness in GTC")

In [ ]:
np.ma.count(cat_lss["G_MAG_LSS"]),np.ma.count(cat_lss["R_MAG_LSS"]), np.ma.count(cat_lss["I_MAG_LSS"])

In [ ]:
plot_completeness_vs_mag(cat_x_lss, "I_MAG_LSS", "I_MAG", bins=20, label="I")
plt.legend()
plt.xlabel("LSS magnitude")
plt.ylabel("completeness in GTC")

In [ ]:
plot_completeness_vs_mag(cat_x_ps, "G_MAG", "G_MAG_PS", bins=10, label="G (PS)")
plot_completeness_vs_mag(cat_x_ps, "R_MAG", "R_MAG_PS", bins=10, label="R (PS)")
plot_completeness_vs_mag(cat_x_ps, "I_MAG", "I_MAG_PS", bins=10, label="I (PS)")

plot_completeness_vs_mag(cat_x_lss, "I_MAG", "I_MAG_LSS", bins=10, label="I (LSS)")
plt.legend()
plt.xlabel("GTC magnitude")


In [ ]:
plot_xmatch_residuals(cat_x_lss)

In [ ]:
plot_xmatch_residuals(cat_x_ps)

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat["I_MAG"], bins, histtype="step", label="osiris")

plt.hist(cat_lss["I_MAG_LSS"], bins, histtype="step", label="LSS")
plt.hist(cat_ps["I_MAG_PS"], bins, histtype="step", label="PS")
plt.xlabel("r mag")
plt.ylabel("count")
arya.Legend(-1)

## Residuals between catalogues

In [ ]:
plot_mag_comparison(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_LSS_ERR",
                    filters=["g", "i"],
                    ref_label="osiris",
                    cmp_label="LSS"
                    
                   )

In [ ]:
plot_mag_2d_residual(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    filters=["i", "r"])


In [ ]:
plot_mag_comparison(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_2d_residual(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                   )

In [ ]:
plot_mag_residual_hist(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], 
                       xlim=(-10, 10),
                       sigma_scale=2
                   )

In [ ]:
plot_mag_residual_hist(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_LSS_ERR", 
                    filters=["R", "I"], 
                       xlim=(-10, 10),
                       sigma_scale=0
                   )

In [ ]:
params = get_default_params("yasone2")


In [ ]:
detection_shifts(cat_x_ps, params, gcol_2="G_MAG_PS", rcol_2="R_MAG_PS", icol_2="I_MAG_PS")

## Magnitude methods

## Comparing photometry methods

### Forced Photometry

In [ ]:
params = get_default_params("yasone2")

In [ ]:
cat_sep = ana_utils.read_catalogue("yasone2", catname="julen")
cat_julen = ana_utils.read_catalogue("yasone2", catname="allcolours_julen_stack")

cat_forced = ana_utils.read_catalogue("yasone2", catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat_psf = ana_utils.read_catalogue("yasone2", catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_forced = ana_utils.read_catalogue("yasone2", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")



In [ ]:
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_3")
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_LB_3")

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF_ANA")

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=1
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                   )

In [ ]:
detection_shifts(cat_forced, params,  gcol_2="G_MED_MAG_APER_3", rcol_2="R_MED_MAG_APER_3", icol_2="I_MED_MAG_APER_3")


### Forced photometry (local background)

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="local bkg"
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                   )

In [ ]:
detection_shifts(cat_forced, params, gcol_1="G_MED_MAG_APER_3", rcol_1="R_MED_MAG_APER_3", icol_1="I_MED_MAG_APER_3",  
                 gcol_2="G_MED_MAG_APER_LB_3", rcol_2="R_MED_MAG_APER_LB_3", icol_2="I_MED_MAG_APER_LB_3")


In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"],
                   )

### PSF

In [ ]:
plot_mag_comparison(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"], ref_label="aperture stack", cmp_label="forced PSF"
                   )

In [ ]:
plot_mag_residual_hist(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=1
                   )

In [ ]:
detection_shifts(cat_psf, params, gcol_2="G_MED_MAG_PSF", rcol_2="R_MED_MAG_PSF", icol_2="I_MED_MAG_PSF")


### SEP

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_sep, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_joined, "yasone2");

In [ ]:
cat_joined["G_MAG_2"] += 0.1
cat_joined["R_MAG_2"] += 0.3
cat_joined["I_MAG_2"] += 0.35

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
params_new = copy(params)
params_new.detection_sigma = None

In [ ]:
detection_shifts(cat_joined, params_new, )

### Julen's data / process

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_julen, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_joined, "yasone2");

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
params_new = copy(params)
params_new.flags_max = None
params_new.flags_weight_max = None
params_new.flags_iso_max = None
params_new.detection_sigma = None

In [ ]:
detection_shifts(cat_joined, params_new,)

# Yasone 3

In [ ]:
cat = ana_utils.read_catalogue("yasone3", catname="allcolours", shiftname="allcolours_panstarrs_shift")

cat_ps = get_panstarrs("yasone3")
cat_lss = get_lss_cat("yasone3")
cat = filter_edges(cat, "yasone3")
cat_ps = filter_edges(cat_ps, "yasone3")
cat_lss = filter_edges(cat_lss, "yasone3")

In [ ]:
cat_x_lss = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
cat_x_ps = phot_utils.outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_x_lss, "yasone3")
fix_coordinates(cat_x_ps, "yasone3")

## Comparing the completeness of each catalogue

In [ ]:
plot_completeness_vs_mag(cat_x_ps, "G_MAG_PS", "G_MAG", bins=20, label="G")
plot_completeness_vs_mag(cat_x_ps, "R_MAG_PS", "R_MAG", bins=20, label="R")
plot_completeness_vs_mag(cat_x_ps, "I_MAG_PS", "I_MAG", bins=20, label="I")
plt.legend()
plt.xlabel("PS magnitude")
plt.ylabel("completeness in GTC")

In [ ]:
np.ma.count(cat_lss["G_MAG_LSS"]),np.ma.count(cat_lss["R_MAG_LSS"]), np.ma.count(cat_lss["I_MAG_LSS"])

In [ ]:
plot_completeness_vs_mag(cat_x_lss, "G_MAG_LSS", "G_MAG", bins=20, label="I")
plt.legend()
plt.xlabel("LSS magnitude")
plt.ylabel("completeness in GTC")

In [ ]:
plot_completeness_vs_mag(cat_x_ps, "G_MAG", "G_MAG_PS", bins=10, label="G (PS)")
plot_completeness_vs_mag(cat_x_ps, "R_MAG", "R_MAG_PS", bins=10, label="R (PS)")
plot_completeness_vs_mag(cat_x_ps, "I_MAG", "I_MAG_PS", bins=10, label="I (PS)")

plot_completeness_vs_mag(cat_x_lss, "G_MAG", "G_MAG_LSS", bins=10, label="G (LSS)")
plt.legend()
plt.xlabel("GTC magnitude")


In [ ]:
plot_xmatch_residuals(cat_x_lss)

In [ ]:
plot_xmatch_residuals(cat_x_ps)

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat["G_MAG"], bins, histtype="step", label="osiris")

plt.hist(cat_lss["G_MAG_LSS"], bins, histtype="step", label="LSS")
plt.hist(cat_ps["G_MAG_PS"], bins, histtype="step", label="PS")
plt.xlabel("g mag")
plt.ylabel("count")
arya.Legend(-1)

## Residuals between catalogues

In [ ]:
plot_mag_comparison(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_LSS_ERR",
                    filters=["g", "i"],
                    ref_label="osiris",
                    cmp_label="LSS"
                    
                   )

In [ ]:
plot_mag_2d_residual(cat_x_lss, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_LSS", 
                    filters=["g", "r"])


In [ ]:
plot_mag_comparison(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_2d_residual(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                   )

In [ ]:
plot_mag_residual_hist(cat_x_ps, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], 
                       xlim=(-10, 10),
                       sigma_scale=2
                   )

In [ ]:
params = get_default_params("yasone3")


In [ ]:
detection_shifts(cat_x_ps, params, gcol_2="G_MAG_PS", rcol_2="R_MAG_PS", icol_2="I_MAG_PS")

## Comparing photometry systems

### Forced Photometry

In [ ]:
params = get_default_params("yasone3")

In [ ]:
cat_sep = ana_utils.read_catalogue("yasone3", catname="julen")
cat_julen = ana_utils.read_catalogue("yasone3", catname="allcolours_julen_stack")

cat_forced = ana_utils.read_catalogue("yasone3", catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat_psf = ana_utils.read_catalogue("yasone3", catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_forced = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")



In [ ]:
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_3")
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_LB_3")

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF_ANA")

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=1
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                   )

In [ ]:
detection_shifts(cat_forced, params,  gcol_2="G_MED_MAG_APER_3", rcol_2="R_MED_MAG_APER_3", icol_2="I_MED_MAG_APER_3")


### Forced photometry (local background)

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="local bkg"
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                   )

In [ ]:
detection_shifts(cat_forced, params, gcol_1="G_MED_MAG_APER_3", rcol_1="R_MED_MAG_APER_3", icol_1="I_MED_MAG_APER_3",  
                 gcol_2="G_MED_MAG_APER_LB_3", rcol_2="R_MED_MAG_APER_LB_3", icol_2="I_MED_MAG_APER_LB_3")


In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"],
                   )

### PSF

In [ ]:
plot_mag_comparison(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"], ref_label="aperture stack", cmp_label="forced PSF"
                   )

In [ ]:
plot_mag_residual_hist(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=1
                   )

In [ ]:
detection_shifts(cat_psf, params, gcol_2="G_MED_MAG_PSF", rcol_2="R_MED_MAG_PSF", icol_2="I_MED_MAG_PSF")


### SEP

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_sep, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_joined, "yasone3");

In [ ]:
cat_joined["G_MAG_2"] += 0.1
cat_joined["R_MAG_2"] += 0.3
cat_joined["I_MAG_2"] += 0.35

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
params_new = copy(params)
params_new.detection_sigma = None

In [ ]:
detection_shifts(cat_joined, params_new, )

### Julen's data / process

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_julen, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
fix_coordinates(cat_joined, "yasone3");

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    xlim=(28, 18), ylim_mag=(28, 18), )

In [ ]:
params_new = copy(params)
params_new.flags_max = None
params_new.flags_weight_max = None
params_new.flags_iso_max = None
params_new.detection_sigma = None

In [ ]:
detection_shifts(cat_joined, params_new,)